In [1]:
# ==========================================
# CELL 1: DEPENDENCIES & ENVIRONMENT SETUP
# ==========================================
!pip install -q gradio pandas numpy pillow torch transformers accelerate opencv-python matplotlib

import os
import sys
import torch
import cv2
import pandas as pd
import numpy as np
from PIL import Image

print(f"✅ AquaCheck Engine Initialized.")
print(f"PyTorch Version: {torch.__version__} | CUDA Active: {torch.cuda.is_available()}")

✅ AquaCheck Engine Initialized.
PyTorch Version: 2.10.0+cu128 | CUDA Active: True


In [2]:
# ==========================================
# CELL 2: CV COLOR & TURBIDITY ANALYSIS ENGINE
# ==========================================
import cv2
import numpy as np
from PIL import Image

def analyze_uploaded_image(image_array):
    """
    Analyzes an uploaded photo for turbidity (cloudiness) 
    and dominant color hue (for test strip or discolored water).
    """
    if image_array is None:
        return "No image uploaded", 0.0, "Clear / Normal"
    
    # Convert image to OpenCV format
    img = cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR)
    
    # Calculate Laplacian variance for edge sharpess / cloudiness
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    # Turbidity heuristic: very low variance or high uniform haze = cloudy
    avg_brightness = np.mean(gray)
    if variance < 100 and avg_brightness > 120:
        detected_clarity = "Cloudy / Murky"
    else:
        detected_clarity = "Clear / Normal"
        
    # Analyze dominant color in HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    avg_hue = np.mean(hsv[:, :, 0]) # 0-180 scale
    
    return f"Image Processed (Avg Hue: {int(avg_hue)}, Sharpness: {int(variance)})", avg_hue, detected_clarity

print("✅ Computer Vision (CV) calibration engine ready.")

✅ Computer Vision (CV) calibration engine ready.


In [3]:
# ==========================================
# CELL 3: MULTI-TRANSLATION & TRIAGE LOGIC
# ==========================================
import json
import pandas as pd
from datetime import datetime

SESSION_LOGS = [
    ["10:15:20", "North District", 85, "SAFE", "None"],
    ["10:22:10", "River Basin B", 42, "HIGH", "Rotten Egg Odor, High Turbidity"],
    ["10:45:00", "Village Well #4", 65, "MODERATE", "High Acidic pH"]
]

# Multilingual Treatment Translations
TRANSLATIONS = {
    "English": {
        "boil": "1. **Rolling Boil:** Boil water rapidly for at least 1 full minute (3 mins at high altitude).",
        "bleach": "2. **Household Bleach:** Add 2 drops of unscented bleach per liter. Wait 30 minutes.",
        "filter": "3. **Pre-Filter:** Pour water through a clean cotton cloth before boiling to remove sediment."
    },
    "Spanish": {
        "boil": "1. **Hervir:** Hierva el agua a fuego vivo durante al menos 1 minuto completo.",
        "bleach": "2. **Cloro:** Agregue 2 gotas de blanqueador sin aroma por litro. Espere 30 minutos.",
        "filter": "3. **Pre-filtrado:** Cuele el agua a través de un paño limpio para quitar la tierra."
    },
    "Swahili": {
        "boil": "1. **Chemsha:** Chemsha maji kwa dakika 1 nzima ili kuua wadudu.",
        "bleach": "2. **Dawa ya Maji:** Weka matone 2 ya dawa kwa kila lita moja. Subiri dakika 30.",
        "filter": "3. **Chuja:** Chuja maji kwa kitambaa safi kabla ya kuchemsha."
    },
    "French": {
        "boil": "1. **Faire Bouillir:** Portez l'eau à ébullition pendant au moins 1 minute.",
        "bleach": "2. **Eau de Javel:** Ajoutez 2 gouttes de javel non parfumée par litre. Attendez 30 minutes.",
        "filter": "3. **Pré-filtration:** Filtrez l'eau à travers un tissu propre avant de la traiter."
    }
}

def analyze_water_sample(odor, clarity, test_strip, symptoms, location_name, target_lang, uploaded_img):
    """Runs vision check, computes WQI score, and localized treatment instructions."""
    
    cv_info, avg_hue, auto_clarity = analyze_uploaded_image(uploaded_img)
    
    # If user provided image, auto-adjust clarity if discrepancy detected
    final_clarity = clarity if uploaded_img is None else auto_clarity
    
    risk_score = 0
    detected_issues = []
    
    if odor != "None / Normal":
        risk_score += 35
        detected_issues.append(f"{odor} Odor")
        
    if final_clarity != "Clear / Normal":
        risk_score += 30
        detected_issues.append("High Turbidity / Suspended Particles")
        
    if test_strip in ["High Acidic (pH < 6.0)", "High Nitrates / Chlorine"]:
        risk_score += 25
        detected_issues.append("Chemical / pH Imbalance")
        
    if symptoms != "None":
        risk_score += 40
        detected_issues.append(f"Reported Symptoms ({symptoms})")
        
    wqi = max(10, 100 - risk_score)
    
    if wqi >= 80:
        risk_level = "SAFE"
        color_code = "#10b981"
        badge_glow = "rgba(16, 185, 129, 0.3)"
    elif wqi >= 50:
        risk_level = "MODERATE"
        color_code = "#f59e0b"
        badge_glow = "rgba(245, 158, 11, 0.3)"
    else:
        risk_level = "HIGH"
        color_code = "#ef4444"
        badge_glow = "rgba(239, 68, 68, 0.3)"
        
    issues_str = ", ".join(detected_issues) if detected_issues else "None Identified"
    
    # Get translation strings
    lang_dict = TRANSLATIONS.get(target_lang, TRANSLATIONS["English"])
    treatment_markdown = f"""
### 🧰 DIY Water Purification Guide ({target_lang})
* {lang_dict['filter']}
* {lang_dict['boil']}
* {lang_dict['bleach']}
    """
    
    # Glassmorphic Output Card
    card_html = f"""
    <div style="background: rgba(30, 41, 59, 0.4); backdrop-filter: blur(12px); border: 1px solid {color_code}; border-radius:16px; padding:24px; color:#fff; box-shadow: 0 10px 30px {badge_glow};">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:14px;">
            <span style="font-size:0.8rem; color:#94a3b8; text-transform:uppercase; letter-spacing:0.08em; font-weight:600;">Water Quality Index</span>
            <span style="background:{color_code}; color:#fff; padding:6px 16px; border-radius:20px; font-weight:700; font-size:0.85rem; box-shadow:0 0 12px {color_code};">
                {risk_level} RISK
            </span>
        </div>
        
        <div style="font-size:2.8rem; font-weight:800; color:#f8fafc; margin-bottom:6px;">
            {wqi} <span style="font-size:1.1rem; color:#94a3b8; font-weight:400;">/ 100 WQI</span>
        </div>
        
        <p style="margin:0 0 16px 0; font-size:0.95rem; color:#cbd5e1;">📍 Location: <strong>{location_name}</strong></p>
        
        <div style="background:rgba(15,23,42,0.6); padding:14px; border-radius:10px; border:1px solid rgba(255,255,255,0.08);">
            <div style="font-size:0.75rem; color:#94a3b8; text-transform:uppercase; font-weight:600;">CV & Triage Findings</div>
            <div style="font-size:0.9rem; font-weight:500; color:#f8fafc; margin-top:4px;">{issues_str}</div>
            <div style="font-size:0.8rem; color:#64748b; margin-top:6px;">Vision Status: {cv_info}</div>
        </div>
    </div>
    """
    
    # Log Entry
    timestamp = datetime.now().strftime("%H:%M:%S")
    SESSION_LOGS.append([timestamp, location_name, wqi, risk_level, issues_str])
    df = pd.DataFrame(SESSION_LOGS, columns=["Time", "Location", "WQI", "Risk Level", "Primary Issue"])
    
    csv_file = "water_assessment_report.csv"
    df.to_csv(csv_file, index=False)
    
    payload = {
        "timestamp": timestamp,
        "location": location_name,
        "wqi": wqi,
        "risk_level": risk_level,
        "issues": detected_issues,
        "vision_metrics": {"avg_hue": avg_hue, "clarity": auto_clarity}
    }
    
    return card_html, treatment_markdown, json.dumps(payload, indent=2), df, csv_file

print("✅ Backend translation & triage logic ready.")

✅ Backend translation & triage logic ready.


In [4]:
# ==========================================
# CELL 4: MODERN GLASSMORPHIC GRADIO APP
# ==========================================
import gradio as gr

APP_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap');

.gradio-container {
    background: #080c14 !important;
    font-family: 'Plus Jakarta Sans', sans-serif !important;
    max-width: 1280px !important;
    margin: 0 auto !important;
    color: #f8fafc !important;
}

.app-header {
    text-align: center;
    padding: 28px 16px;
    background: rgba(30, 41, 59, 0.25) !important;
    backdrop-filter: blur(16px) !important;
    border: 1px solid rgba(255, 255, 255, 0.08) !important;
    border-radius: 20px !important;
    margin-bottom: 24px !important;
    box-shadow: 0 20px 40px rgba(0,0,0,0.4) !important;
}

.glass-card {
    background: rgba(15, 23, 42, 0.6) !important;
    backdrop-filter: blur(12px) !important;
    border: 1px solid rgba(255, 255, 255, 0.08) !important;
    border-radius: 16px !important;
    padding: 24px !important;
    margin-bottom: 16px !important;
}

.primary-btn {
    background: linear-gradient(135deg, #0284c7 0%, #2563eb 100%) !important;
    color: #ffffff !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
    border: none !important;
    padding: 14px !important;
    font-size: 1rem !important;
    box-shadow: 0 4px 20px rgba(37, 99, 235, 0.4) !important;
    transition: all 0.2s ease !important;
}

.primary-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 24px rgba(37, 99, 235, 0.6) !important;
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=APP_CSS, title="AquaCheck Edge - Safe Water AI") as demo:
    
    # App Header
    with gr.Row(elem_classes=["app-header"]):
        gr.HTML("""
            <div>
                <h1 style="margin:0; font-size:2rem; font-weight:800; color:#ffffff; letter-spacing:-0.02em;">
                    💧 AquaCheck Edge
                </h1>
                <p style="margin:6px 0 0 0; font-size:1rem; color:#94a3b8;">
                    AI-Powered Water Safety Analysis & Field Treatment Guide
                </p>
            </div>
        """)

    with gr.Tabs():
        
        # TAB 1: SMART TRIAGE
        with gr.Tab("🚰 Smart Assessment"):
            with gr.Row(equal_height=False):
                
                # Left Column: Inputs
                with gr.Column(scale=6, elem_classes=["glass-card"]):
                    gr.Markdown("### 🔍 Water Sample Observation")
                    
                    with gr.Row():
                        location_input = gr.Textbox(label="📍 Location / Village Name", value="Local Neighborhood", scale=3)
                        gps_btn = gr.Button("📍 GPS", scale=1, size="sm")

                    with gr.Row():
                        clarity_input = gr.Dropdown(
                            choices=["Clear / Normal", "Cloudy / Murky", "Dirty / Discolored"],
                            value="Clear / Normal",
                            label="1. Visual Clarity"
                        )
                        odor_input = gr.Dropdown(
                            choices=["None / Normal", "Musty / Earthy", "Rotten Egg (Sulfur)", "Chemical / Metallic"],
                            value="None / Normal",
                            label="2. Water Odor"
                        )

                    with gr.Row():
                        test_strip_input = gr.Dropdown(
                            choices=["Normal / Neutral", "High Acidic (pH < 6.0)", "High Nitrates / Chlorine", "Not Tested"],
                            value="Not Tested",
                            label="3. Test Strip Result"
                        )
                        symptoms_input = gr.Dropdown(
                            choices=["None", "Stomach Pain / Diarrhea", "Skin Rash / Itching", "Fever / Vomiting"],
                            value="None",
                            label="4. Health Symptoms"
                        )

                    image_upload = gr.Image(label="📷 Upload Photo of Water Glass / Test Strip (Auto CV)", type="numpy")
                    lang_choice = gr.Radio(["English", "Spanish", "Swahili", "French"], value="English", label="🌐 Advice Language")
                    
                    check_btn = gr.Button("⚡ Analyze Water Safety", variant="primary", elem_classes=["primary-btn"])

                # Right Column: Output & Action Plan
                with gr.Column(scale=6, elem_classes=["glass-card"]):
                    gr.Markdown("### 📊 Safety Analysis & Action Plan")
                    
                    results_card = gr.HTML("""
                        <div style="text-align:center; padding: 50px 20px; border: 1px dashed rgba(255,255,255,0.15); border-radius:14px;">
                            <div style="font-size:3rem; margin-bottom:10px;">🚰</div>
                            <div style="color:#94a3b8; font-size:1rem;">Fill in the sample details on the left and tap 'Analyze Water Safety'.</div>
                        </div>
                    """)
                    
                    diy_treatment = gr.Markdown("### 🧰 DIY Home Treatment Steps\n*Instructions will appear after assessment.*")
                    
                    with gr.Accordion("📂 Export Report & Technical Payload", open=False):
                        file_download = gr.File(label="📥 Download CSV Log")
                        json_debug = gr.Code(label="Raw JSON Response", language="json")

        # TAB 2: COMMUNITY LOGS & MONITORING
        with gr.Tab("📋 Community Field Logs"):
            with gr.Column(elem_classes=["glass-card"]):
                gr.Markdown("### 📊 Community Water Safety History")
                history_df = gr.Dataframe(
                    value=[
                        ["10:15:20", "North District", 85, "SAFE", "None"],
                        ["10:22:10", "River Basin B", 42, "HIGH", "Rotten Egg Odor, High Turbidity"],
                        ["10:45:00", "Village Well #4", 65, "MODERATE", "High Acidic pH"]
                    ],
                    headers=["Time", "Location", "WQI", "Risk Level", "Primary Issue"],
                    interactive=False
                )

        # TAB 3: EMERGENCY PROTOCOLS
        with gr.Tab("🛡️ Emergency Guidelines"):
            with gr.Column(elem_classes=["glass-card"]):
                gr.Markdown("""
                ### 🚨 Critical Water Emergency Actions
                If severe contamination or sewage leak is suspected:
                * **DO NOT** drink, wash food, or bathe in untreated water.
                * **Boiling Rule:** Bring to a full rolling boil for at least **1 minute** (3 minutes at high elevations).
                * **Disinfection:** Use 8 drops of 6% unscented bleach per gallon of clear water if boiling fuel is unavailable.
                * **Contact:** Report contamination immediately to local municipal authorities or emergency health workers.
                """)

    # Event Bindings
    gps_btn.click(
        lambda: "District 4 (GPS: 12.9716° N, 77.5946° E)",
        outputs=location_input
    )

    check_btn.click(
        fn=analyze_water_sample,
        inputs=[odor_input, clarity_input, test_strip_input, symptoms_input, location_input, lang_choice, image_upload],
        outputs=[results_card, diy_treatment, json_debug, history_df, file_download]
    )

demo.launch(share=True)

/tmp/ipykernel_23/3917619068.py:55: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=APP_CSS, title="AquaCheck Edge - Safe Water AI") as demo:
/tmp/ipykernel_23/3917619068.py:55: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=APP_CSS, title="AquaCheck Edge - Safe Water AI") as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b725d7572c7feb46ea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
